# Assignment 6: Transformer Fine-tuning

BERT, DistilBERT, and RoBERTa for Sentiment Classification

In [ ]:

# Install dependencies
# !pip install transformers datasets accelerate torch scikit-learn matplotlib seaborn bertviz


In [ ]:

from datasets import load_dataset

dataset = load_dataset("sst2")
train_data = dataset["train"]
val_data = dataset["validation"]
print(dataset)


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(train_data)
df['label'].value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.show()


In [ ]:

from transformers import BertTokenizer, BertForSequenceClassification

model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


In [ ]:

def tokenize_function(examples):
    return tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_val = val_data.map(tokenize_function, batched=True)


In [ ]:

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [ ]:

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)


In [ ]:

# trainer.train()
# trainer.evaluate()


In [ ]:

# DistilBERT

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

distil_tokenizer = DistilBertTokenizer.from_pretrained(
    'distilbert-base-uncased'
)

distil_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)


In [ ]:

# RoBERTa

from transformers import RobertaTokenizer, RobertaForSequenceClassification

roberta_tokenizer = RobertaTokenizer.from_pretrained(
    'roberta-base'
)

roberta_model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=2
)


In [ ]:

# Attention Visualization

from transformers import BertForSequenceClassification

attention_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    output_attentions=True
)



## Required Experiments

- Learning rates: 1e-5, 2e-5, 3e-5, 5e-5
- Batch sizes: 8, 16, 32
- Epochs: 2, 3, 4, 5
- Sequence lengths: 128, 256, 512

## Comparison Table
- BERT
- DistilBERT
- RoBERTa
- LSTM (Assignment 5)

## Advanced Techniques
- Layer-wise learning rate decay
- Gradual unfreezing
- Data augmentation
- Ensemble methods
- Knowledge distillation
- Mixed precision (FP16)

## Deployment
- ONNX export
- Streamlit inference app
